# TFG Crypto — Detección de regímenes de mercado (PCA + k-means + HMM)

Notebook **independiente** de la preparación de la LSTM y del RAG. Aquí no predecimos:
**clasificamos cada día en un régimen de mercado** (tranquilo / alcista / bajista-volátil...)
para entender la no-estacionariedad del mercado y, opcionalmente, generar una etiqueta de
régimen que pueda alimentar a la LSTM o al RAG.

Orden:
1. **Setup** y carga del mismo `df_merged.csv`.
2. **Variables de régimen** — pocas y limpias (volatilidad, tendencia, caída).
3. **PCA** — para visualizar en 2D (lo que ya conoces).
4. **k-means** — agrupa, pero SIN memoria temporal (parpadea).
5. **HMM** — agrupa CON memoria: los regímenes son pegajosos, no parpadea.
6. **Ventana 14d vs 30d** — el compromiso entre detectar rápido o detectar fiable.
7. **Aviso de lookahead** — versión descriptiva vs versión causal.

**Idea de fondo:** un régimen dura meses. Si ayer era bear, hoy lo más probable es que siga
siendo bear aunque los síntomas de hoy sean ambiguos. El HMM modela justo eso.

## 1. Setup y carga de datos

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from hmmlearn.hmm import GaussianHMM

pd.set_option("display.max_columns", None)

# Ajusta la ruta si tu CSV está en otro sitio (misma que en el notebook de preparación)
RUTA_CSV = "../data/csv/df_merged.csv"

df = pd.read_csv(RUTA_CSV, parse_dates=["date"], index_col="date").sort_index()
print(f"Shape: {df.shape}  |  Rango: {df.index.min().date()} -> {df.index.max().date()}")
df.head(3)

## 2. Variables que definen un régimen

Un régimen NO necesita las 57 features. Se describe con pocas cosas:
- **Volatilidad** (14 y 30 días): cuánto se mueve el mercado.
- **Tendencia**: hacia dónde va (distancia a la media móvil de 50 días, retorno acumulado a 30d).
- **Caída desde máximos** (drawdown): cómo de hundido está respecto a su techo.

**Importante:** para el HMM/k-means usaremos un set SIN redundancia. Meter a la vez `vol_14d`
y `vol_30d` (que dicen casi lo mismo) confunde al HMM y lo hace parpadear. Calculamos las dos
para el análisis de la ventana (sección 6), pero al clasificar usamos solo `vol_30d`.

In [ ]:
eth = df["eth_close"]
ret = eth.pct_change() * 100                       # retorno % diario de ETH

reg = pd.DataFrame(index=df.index)
reg["ret"]          = ret
reg["vol_14d"]      = ret.rolling(14).std()        # volatilidad rápida (nerviosa)
reg["vol_30d"]      = ret.rolling(30).std()        # volatilidad lenta (estable)
reg["cum_ret_30d"]  = ret.rolling(30).sum()        # tendencia: ¿venimos subiendo o bajando?
reg["dist_sma50"]   = (eth / eth.rolling(50).mean() - 1) * 100   # régimen alcista (>0) / bajista (<0)
reg["drawdown"]     = (eth / eth.cummax() - 1) * 100             # posición en el ciclo (0=máximos, -80=suelo)
reg["precio"]       = eth

reg = reg.dropna().copy()
print(f"Días con datos completos: {len(reg)}  ({reg.index.min().date()} -> {reg.index.max().date()})")

# Variables que entran a la clasificación (SIN redundancia: solo vol_30d)
VARS_REGIMEN = ["vol_30d", "cum_ret_30d", "dist_sma50", "drawdown"]
print(f"Variables para clasificar regímenes: {VARS_REGIMEN}")
reg[VARS_REGIMEN].describe().T

## 3. PCA: ver los datos en 2D

Estandarizamos (media 0, desviación 1) y proyectamos a 2 dimensiones para poder pintar.
Es exactamente lo que hiciste con los clientes: PCA no clasifica, solo permite *ver* si hay
estructura. Si los puntos forman nubes distinguibles, hay regímenes que separar.

In [ ]:
X = StandardScaler().fit_transform(reg[VARS_REGIMEN].values)

pca = PCA(n_components=2).fit(X)
Xp = pca.transform(X)
print(f"Varianza explicada por las 2 componentes: {pca.explained_variance_ratio_.round(3)}  "
      f"(suma {pca.explained_variance_ratio_.sum():.1%})")

plt.figure(figsize=(7, 6))
plt.scatter(Xp[:, 0], Xp[:, 1], s=6, alpha=0.4, c=reg["dist_sma50"].values, cmap="RdYlGn")
plt.colorbar(label="distancia a SMA50 (verde=alcista, rojo=bajista)")
plt.xlabel("Componente principal 1"); plt.ylabel("Componente principal 2")
plt.title("Días de mercado proyectados con PCA"); plt.tight_layout(); plt.show()

## 4. k-means: agrupa, pero SIN memoria temporal

Tu método conocido. Agrupa los días en k clusters por parecido. Probamos k=3.
**El problema que verás:** k-means trata cada día por separado, no sabe que el tiempo existe.
Por eso, al pintar el precio coloreado por cluster, verás que **parpadea** (whipsaw): cambia de
régimen de un día para otro, lo cual no tiene sentido para algo que dura meses.

In [ ]:
km = KMeans(n_clusters=3, n_init=10, random_state=42).fit(X)
reg["cluster_km"] = km.labels_

# Interpretar cada cluster por sus medias reales
print("── Media de cada variable por cluster (k-means) ──")
print(reg.groupby("cluster_km")[VARS_REGIMEN].mean().round(2).to_string())

cambios_km = int((np.diff(reg["cluster_km"].values) != 0).sum())
print(f"\nCambios de etiqueta a lo largo del tiempo (k-means): {cambios_km}")

In [ ]:
# El precio de ETH coloreado por el régimen que asigna k-means
fig, ax = plt.subplots(figsize=(14, 5))
for c in sorted(reg["cluster_km"].unique()):
    m = reg["cluster_km"] == c
    ax.scatter(reg.index[m], reg["precio"][m], s=8, label=f"cluster {c}")
ax.set_yscale("log"); ax.set_ylabel("Precio ETH (escala log)")
ax.set_title(f"Regímenes según k-means  —  {cambios_km} cambios (fíjate en el parpadeo)")
ax.legend(); plt.tight_layout(); plt.show()

## 5. HMM: agrupa CON memoria — los regímenes son pegajosos

El Hidden Markov Model aprende, además de cómo es cada régimen, la **probabilidad de quedarse
en el mismo régimen al día siguiente** (la diagonal de la matriz de transición). Como aprende
que cambiar es raro, no parpadea: da bloques limpios y continuos, y marca de forma natural
dónde empieza y acaba cada régimen.

**Truco de robustez:** el HMM puede caer en soluciones malas según la inicialización. Probamos
varias semillas y nos quedamos con la de mayor verosimilitud (`score`). Esto es clave: sin ello
el HMM puede parpadear incluso más que el k-means.

In [ ]:
def ajustar_mejor_hmm(X, n_estados=3, semillas=range(12)):
    """Prueba varias semillas y devuelve el HMM con mayor verosimilitud."""
    mejor = None
    for s in semillas:
        m = GaussianHMM(n_components=n_estados, covariance_type="full",
                        n_iter=300, random_state=s, tol=1e-4)
        try:
            m.fit(X)
            sc = m.score(X)
        except Exception:
            continue
        if mejor is None or sc > mejor[0]:
            mejor = (sc, m)
    return mejor[1]

hmm = ajustar_mejor_hmm(X, n_estados=3)
reg["estado_hmm"] = hmm.predict(X)

cambios_hmm = int((np.diff(reg["estado_hmm"].values) != 0).sum())
print(f"Cambios de etiqueta (HMM): {cambios_hmm}   vs   k-means: {cambios_km}")
print(f"\nMatriz de transición (fila = estado de hoy, columna = estado de mañana):")
print(np.round(hmm.transmat_, 3))
print(f"\nPersistencia (prob. de quedarse en el mismo estado): {np.diag(hmm.transmat_).round(3)}")
print("Cuanto más cerca de 1, más pegajoso es el régimen.")

In [ ]:
# Interpretar cada estado del HMM por sus medias reales
print("── Media de cada variable por estado (HMM) ──")
print(reg.groupby("estado_hmm")[VARS_REGIMEN].mean().round(2).to_string())
print("\nLee la tabla así: el estado con vol alta y dist_sma50 muy negativa = bajista-volátil;")
print("el de vol baja y dist_sma50 cerca de 0 = lateral tranquilo; etc.")

In [ ]:
# El precio coloreado por el régimen del HMM: compáralo con el de k-means
fig, ax = plt.subplots(figsize=(14, 5))
for s in sorted(reg["estado_hmm"].unique()):
    m = reg["estado_hmm"] == s
    ax.scatter(reg.index[m], reg["precio"][m], s=8, label=f"estado {s}")
ax.set_yscale("log"); ax.set_ylabel("Precio ETH (escala log)")
ax.set_title(f"Regímenes según HMM  —  {cambios_hmm} cambios (bloques limpios, sin parpadeo)")
ax.legend(); plt.tight_layout(); plt.show()

## 6. La pregunta de la ventana: 14 días vs 30 días

Tu pregunta sobre acertar el principio y el final de un régimen lo antes posible. La ventana
de la volatilidad es la palanca, y hay un compromiso que no tiene solución mágica:

- **14 días:** reacciona **rápido** a un cambio, pero es **nerviosa** → te da falsas alarmas.
- **30 días:** es **suave y fiable**, pero **llega tarde** → cuando avisa, el régimen ya empezó hace 1-2 semanas.

La gráfica lo deja claro: mira cuánto tarda la línea de 30d en reaccionar frente a la de 14d
cuando estalla la volatilidad.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(reg.index, reg["vol_14d"], lw=0.8, label="Volatilidad 14d (rápida, nerviosa)", color="#E08A2B")
ax.plot(reg.index, reg["vol_30d"], lw=1.2, label="Volatilidad 30d (lenta, fiable)", color="#627EEA")
ax.set_ylabel("Volatilidad (desv. típica de retornos %)")
ax.set_title("14d reacciona antes pero salta más; 30d es estable pero va con retraso")
ax.legend(); plt.tight_layout(); plt.show()

# Cuantificar el retraso: en los días de mayor subida de volatilidad, ¿cuánto tarda la 30d?
corr_lag = reg["vol_14d"].corr(reg["vol_30d"])
print(f"Correlación 14d vs 30d: {corr_lag:.3f} (altas, miden lo mismo pero a distinta velocidad)")
print("Por eso al clasificar usamos solo UNA (vol_30d): meter las dos es redundante.")

## 7. Aviso importante: lookahead (mirar el futuro sin querer)

Todo lo anterior ajusta el modelo con los **8 años enteros**. Eso significa que la etiqueta de
régimen del día *t* se calculó sabiendo lo que pasó *después*. Para **describir** la historia y
hacer estas gráficas, es correcto y útil.

**PERO** si quieres usar la etiqueta de régimen como una columna más para la LSTM, tiene que ser
**causal**: calculada usando solo el pasado de cada día. Si no, le estarías colando el futuro al
modelo y el resultado sería tramposo (parecería buenísimo y luego fallaría en real).

Abajo, una versión causal sencilla: ajustamos el HMM SOLO con el primer 70% (train) y con ese
modelo ya fijo etiquetamos el resto. Así la etiqueta de validación/test no usó su propio futuro
para entrenarse. (Nota: `predict` de hmmlearn aún suaviza dentro del tramo; una versión 100%
online día a día sería el siguiente refinamiento, pero esto ya evita la fuga gruesa.)

In [ ]:
n = len(reg)
i_train = int(n * 0.70)
X_train = X[:i_train]

hmm_causal = ajustar_mejor_hmm(X_train, n_estados=3)        # ajustado SOLO con train
reg["estado_hmm_causal"] = hmm_causal.predict(X)            # etiqueta todo con el modelo de train

# ¿Cuánto se parece la versión causal a la descriptiva? (no tienen por qué coincidir los números de estado)
print("Distribución de estados (causal):")
print(reg["estado_hmm_causal"].value_counts().sort_index().to_string())
print("\nEsta columna 'estado_hmm_causal' es la que podrías pasar a la LSTM como feature de régimen,")
print("o al RAG para que diga 'estamos en régimen X, la señal cuantitativa es más/menos fiable'.")

## 8. Guardar resultado

Guardamos el dataframe con las etiquetas de régimen por día, por si quieres reutilizarlo.

In [ ]:
salida = reg[["precio", "vol_30d", "cum_ret_30d", "dist_sma50", "drawdown",
              "cluster_km", "estado_hmm", "estado_hmm_causal"]].copy()
salida.to_csv("../data/csv/regimenes.csv")
print(f"✓ Guardado en ../data/csv/regimenes.csv  ({len(salida)} filas)")
salida.tail()

---
### Cómo leer los resultados

1. **Compara los dos mapas de color** (sección 4 vs 5). Si el del HMM sale en bloques limpios y
   el del k-means parpadea, has demostrado por qué la memoria temporal importa.
2. **Mira la persistencia** (diagonal de la matriz). Si está cerca de 1, el HMM ha aprendido solo
   que los regímenes duran. Ese número es citable en la defensa.
3. **Interpreta los estados** con las tablas de medias: ponles nombre (lateral, alcista, crash...).
4. La columna `estado_hmm_causal` es la candidata a feature de régimen para la LSTM.

**No esperes que esto solo dispare el acierto direccional al 55%.** Es un estudio que explica la
no-estacionariedad del mercado y aporta una herramienta; si además mueve la LSTM, mejor, pero su
valor no depende de eso.